# 手撕 DPO

$$\mathcal{L}_{DPO}(\pi;\pi_{ref}) = -\mathbb{E}_{(a, y_w, y_l)_{\sim}\mathcal{D}}[\log \sigma(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)})-\beta\log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)})]$$

In [1]:
import torch
import torch.nn.functional as F
from transformers import LlamaConfig, LlamaForCausalLM
torch.manual_seed(42)

# 加载模型
config = LlamaConfig(vocab_size = 32,      # default is 32000
                    hidden_size = 256,
                    intermediate_size = 512,
                    num_hidden_layers = 2,
                    num_attention_heads = 4,
                    num_key_value_heads = 4,
                    )
ref_model = LlamaForCausalLM(config)
ref_model.eval()
model = LlamaForCausalLM(config)
print(model.lm_head)

Linear(in_features=256, out_features=32, bias=False)

In [2]:
# Create Preference data
# Chosen :   [Prompt Token,  Response Chosen Token]
# Rejected :   [Prompt Token,  Response Rejected Token]

prompt_length = 6
answer_length = 4
prompt_chosen =   torch.tensor([[5, 8, 9, 10, 5, 3,   16, 29, 18, 17]], dtype=torch.int64)
prompt_rejected = torch.tensor([[5, 8, 9, 10, 5, 3,   26, 14, 31, 0]], dtype=torch.int64)
attention_mask =  torch.tensor([[0, 0, 0, 0,  0, 0,   1,  1,  1,  1]], dtype=torch.bool)

x_chosen = {'input_ids':prompt_chosen, 'attention_mask':attention_mask}
x_rejected = {'input_ids':prompt_chosen, 'attention_mask':attention_mask}

In [3]:
# Calculative Token-Level Policy 

# test for get logits and logprob
output = ref_model(**x_chosen)
print(output.logits.shape)
print(output.logits[0,8,:])
print(output.logits.shape)

# how DPO get target policy
# output.logits.log_softmax(-1)
def get_probs(logits, labels):
    per_token_logps = torch.gather(logits.log_softmax(-1), dim=2, 
                                   index=labels.unsqueeze(2)).squeeze(2)
    return per_token_logps
    
probs_chosen = get_probs(output.logits, prompt_chosen)
print(probs_chosen)

torch.Size([1, 10, 32])

tensor([-0.3935, -0.2199, -0.1951,  0.1714, -0.5579, -0.0424,  0.0545,  0.0262,
         0.2163, -0.3165,  0.1775,  0.5048, -0.0364,  0.2731,  0.0074,  0.8802,
        -0.2099,  0.1268,  0.1451, -0.3297,  0.6911,  0.2241,  0.0662, -0.1679,
         0.0165, -0.1932,  0.0224, -0.1943, -0.2393, -0.0765, -0.2669, -0.0356],
       grad_fn=<SliceBackward0>)

torch.Size([1, 10, 32])

tensor([[-3.7388, -3.1884, -3.1442, -3.3731, -3.7379, -3.0073, -3.5660, -3.7940,
         -3.3733, -3.4868]], grad_fn=<SqueezeBackward1>)

In [4]:
# Show how to get policy by  

labels = prompt_chosen
labels[labels == 0] = 0
print(output.logits.log_softmax(-1))
print(labels)
per_token_logps = torch.gather(output.logits.log_softmax(-1), dim=2, 
                               index=labels.unsqueeze(2)).squeeze(2)
print(per_token_logps)

# 假设取idx=8的token， 那么拿到的token号为18
# 在idx=8的 32 logits 里面取 第18个logits， 即为 PI
idx = labels[0,8]
print(idx)
print(output.logits.log_softmax(-1)[0,8,:])
print(per_token_logps[0,8])

tensor([[[-3.5335, -3.5129, -3.7714, -3.2127, -3.7783, -3.7388, -3.4562,
          -3.6560, -3.1134, -3.2314, -3.2241, -3.5807, -3.7173, -3.2986,
          -3.7240, -3.3236, -3.5979, -3.1992, -3.2744, -3.5812, -2.9747,
          -3.5238, -3.0925, -3.8196, -3.2623, -3.4963, -3.3308, -3.5869,
          -3.8415, -3.6927, -3.9175, -3.8801],
         [-3.4976, -3.3398, -3.6529, -3.2243, -3.8891, -3.5515, -3.5127,
          -3.3787, -3.1884, -3.3081, -3.3454, -3.6040, -3.8475, -3.4015,
          -3.7195, -3.2985, -3.4773, -3.3967, -3.2884, -3.6509, -3.0687,
          -3.5846, -2.9858, -3.6278, -3.2938, -3.4841, -3.4477, -3.6186,
          -3.9043, -3.5219, -3.8240, -3.7958],
         [-3.6848, -3.4321, -3.6031, -3.0608, -4.0866, -3.5365, -3.4072,
          -3.4279, -3.0781, -3.1442, -3.4926, -3.4771, -3.6514, -3.2918,
          -3.6278, -3.6051, -3.5022, -3.4603, -3.1477, -3.6883, -2.9965,
          -3.5996, -3.1912, -3.7750, -3.2990, -3.5582, -3.5868, -3.3334,
          -3.8212, -3.6786, -3.8016, -3.8878],
         [-3.6155, -3.1548, -3.6288, -3.1712, -3.8028, -3.7140, -3.4776,
          -3.7240, -3.2199, -3.2064, -3.3731, -3.6377, -3.5571, -3.3853,
          -3.6592, -3.1945, -3.4831, -3.3551, -3.2366, -3.7928, -3.2831,
          -3.5283, -3.1158, -3.9549, -3.2292, -3.3590, -3.4872, -3.3784,
          -4.0087, -3.5449, -3.7411, -3.7970],
         [-3.5311, -3.5183, -3.7864, -3.1968, -3.7750, -3.7379, -3.4576,
          -3.6493, -3.1260, -3.2332, -3.2191, -3.5675, -3.7114, -3.3079,
          -3.7169, -3.3243, -3.5902, -3.1986, -3.2892, -3.5799, -2.9931,
          -3.5104, -3.1021, -3.8094, -3.2704, -3.4896, -3.3222, -3.5798,
          -3.8400, -3.6871, -3.9138, -3.8775],
         [-3.5684, -3.4234, -3.6725, -3.0073, -3.7951, -3.5821, -3.6909,
          -3.2552, -3.1045, -3.3467, -3.3883, -3.5774, -3.6842, -3.2220,
          -3.8000, -3.3926, -3.5943, -3.2555, -3.3142, -3.6919, -3.1616,
          -3.7013, -3.1477, -3.7128, -3.1729, -3.5214, -3.4141, -3.4360,
          -3.8956, -3.6039, -3.8317, -3.8974],
         [-3.5140, -3.8414, -3.8578, -3.2113, -3.8398, -4.0878, -3.1885,
          -4.0577, -3.3527, -3.7639, -3.1837, -3.3300, -3.2298, -3.3192,
          -3.2483, -2.5343, -3.5660, -3.4686, -3.7017, -3.7164, -3.2095,
          -3.5200, -3.3726, -3.7162, -3.2263, -3.4150, -3.1726, -3.8657,
          -3.6752, -3.7004, -4.0907, -3.8170],
         [-3.9951, -3.6777, -3.7030, -3.2516, -4.0218, -3.9451, -3.3303,
          -3.9284, -3.3701, -3.7283, -3.1522, -2.8708, -3.2721, -3.2742,
          -3.3646, -2.7925, -3.6172, -3.4997, -3.7001, -3.8081, -2.9434,
          -3.3853, -3.4481, -3.7190, -3.4011, -3.2051, -3.3302, -3.5824,
          -3.7199, -3.7940, -3.8977, -3.9288],
         [-3.9120, -3.7383, -3.7135, -3.3471, -4.0763, -3.5608, -3.4639,
          -3.4923, -3.3022, -3.8350, -3.3410, -3.0136, -3.5549, -3.2454,
          -3.5111, -2.6382, -3.7284, -3.3916, -3.3733, -3.8482, -2.8273,
          -3.2943, -3.4523, -3.6863, -3.5020, -3.7116, -3.4961, -3.7127,
          -3.7578, -3.5950, -3.7853, -3.5541],
         [-3.8325, -3.6247, -3.6051, -3.2366, -4.0361, -3.5398, -3.6969,
          -3.5102, -3.2730, -3.5918, -3.3646, -3.1323, -3.6347, -3.2180,
          -3.4528, -2.9185, -3.9525, -3.4868, -3.3810, -3.7825, -2.6764,
          -3.3795, -3.3151, -3.9127, -3.4457, -3.6982, -3.5014, -3.5541,
          -3.7674, -3.6161, -3.6549, -3.4723]]], grad_fn=<LogSoftmaxBackward0>)

tensor([[ 5,  8,  9, 10,  5,  3, 16, 29, 18, 17]])

tensor([[-3.7388, -3.1884, -3.1442, -3.3731, -3.7379, -3.0073, -3.5660, -3.7940,
         -3.3733, -3.4868]], grad_fn=<SqueezeBackward1>)

tensor(18)

tensor([-3.9120, -3.7383, -3.7135, -3.3471, -4.0763, -3.5608, -3.4639, -3.4923,
        -3.3022, -3.8350, -3.3410, -3.0136, -3.5549, -3.2454, -3.5111, -2.6382,
        -3.7284, -3.3916, -3.3733, -3.8482, -2.8273, -3.2943, -3.4523, -3.6863,
        -3.5020, -3.7116, -3.4961, -3.7127, -3.7578, -3.5950, -3.7853, -3.5541],
       grad_fn=<SliceBackward0>)

tensor(-3.3733, grad_fn=<SelectBackward0>)

In [5]:
# 分别计算 ref/model, chosen/rejected,  logtis/prob value
logits_chosen_ref = ref_model(**x_chosen).logits
logits_rejected_ref = ref_model(**x_rejected).logits
logits_chosen = model(**x_chosen).logits
logits_rejected = model(**x_rejected).logits

probs_chosen_ref = get_probs(logits_chosen_ref, prompt_chosen)
probs_chosen = get_probs(logits_chosen, prompt_chosen)
probs_rejected_ref = get_probs(logits_rejected_ref, prompt_rejected)
probs_rejected = get_probs(logits_rejected, prompt_rejected)

In [6]:
import torch.nn.functional as F

beta = 0.1
pi_logratios = probs_chosen - probs_rejected
ref_logratios = probs_chosen_ref - probs_rejected_ref
logits = pi_logratios - ref_logratios
losses = -F.logsigmoid(beta * logits ) * attention_mask
print(losses)
loss = losses.sum(-1)/attention_mask.sum()
print(loss)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.6631, 0.6664, 0.7183,
         0.7293]], grad_fn=<MulBackward0>)

tensor([0.6943], grad_fn=<DivBackward0>)

In [7]:
# 实际效果